# Global Conflict & Economic Impact Analysis

## Phase 6 - Data Integration & Feature Engineering

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns",None)

In [3]:
conflict_df = pd.read_csv("clean_conflict.csv")

military_df = pd.read_csv("clean_military.csv")

inflation_df = pd.read_csv("clean_inflation.csv")

oil_df = pd.read_csv("oil_analysis.csv")

In [4]:
print(conflict_df.shape)

print(military_df.shape)

print(inflation_df.shape)

print(oil_df.shape)

(2816, 28)
(6650, 4)
(6650, 4)
(41, 5)


In [5]:
print(conflict_df.columns)

print(military_df.columns)

print(inflation_df.columns)

print(oil_df.columns)

Index(['conflict_id', 'location', 'side_a', 'side_a_id', 'side_a_2nd',
       'side_b', 'side_b_id', 'side_b_2nd', 'incompatibility',
       'territory_name', 'year', 'intensity_level', 'cumulative_intensity',
       'type_of_conflict', 'start_date', 'start_prec', 'start_date2',
       'start_prec2', 'ep_end', 'ep_end_date', 'ep_end_prec', 'gwno_a',
       'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd', 'gwno_loc', 'region', 'version'],
      dtype='object')
Index(['Country Name', 'Country Code', 'Year', 'Military_Spending'], dtype='object')
Index(['Country Name', 'Country Code', 'Year', 'Inflation_Rate'], dtype='object')
Index(['Year', 'Average_Oil_Price', 'Oil_Price_Change', 'Rolling_Average',
       'Volatility'],
      dtype='object')


In [6]:
country_mapping = {

    "Russian Federation":"Russia",

    "United States":"United States of America",

    "Iran, Islamic Rep.":"Iran",

    "Egypt, Arab Rep.":"Egypt",

    "Syrian Arab Republic":"Syria",

    "Yemen, Rep.":"Yemen",

    "Venezuela, RB":"Venezuela",

    "Kyrgyz Republic":"Kyrgyzstan",

    "Slovak Republic":"Slovakia",

    "Lao PDR":"Laos"

}

In [7]:
military_df["Country Name"] = military_df[
    "Country Name"
].replace(country_mapping)

inflation_df["Country Name"] = inflation_df[
    "Country Name"
].replace(country_mapping)

In [8]:
conflict_count = (

    conflict_df

    .groupby(

        ["location","year"]

    )

    .size()

    .reset_index(

        name="Conflict_Count"

    )

)

conflict_count.head()

,location,year,Conflict_Count
0,Afghanistan,1978,1
1,Afghanistan,1979,1
2,Afghanistan,1980,1
3,Afghanistan,1981,1
4,Afghanistan,1982,1


In [9]:
conflict_intensity = (

    conflict_df

    .groupby(

        ["location","year"]

    )["intensity_level"]

    .mean()

    .reset_index()

)

In [10]:
conflict_summary = conflict_count.merge(

    conflict_intensity,

    on=["location","year"]

)

conflict_summary.head()

,location,year,Conflict_Count,intensity_level
0,Afghanistan,1978,1,2.0
1,Afghanistan,1979,1,2.0
2,Afghanistan,1980,1,2.0
3,Afghanistan,1981,1,2.0
4,Afghanistan,1982,1,2.0


In [11]:
conflict_summary.rename(

    columns={

        "location":"Country Name",

        "year":"Year",

        "intensity_level":"Average_Intensity"

    },

    inplace=True

)

In [12]:
master = conflict_summary.merge(

    military_df,

    on=["Country Name","Year"],

    how="left"

)

master.head()

,Country Name,Year,Conflict_Count,Average_Intensity,Country Code,Military_Spending
0,Afghanistan,1978,1,2.0,NaN,NaN
1,Afghanistan,1979,1,2.0,NaN,NaN
2,Afghanistan,1980,1,2.0,NaN,NaN
3,Afghanistan,1981,1,2.0,NaN,NaN
4,Afghanistan,1982,1,2.0,NaN,NaN


In [13]:
master = master.merge(

    inflation_df,

    on=["Country Name","Year"],

    how="left"

)

In [14]:
master = master.merge(

    oil_df[
        [
            "Year",

            "Average_Oil_Price",

            "Oil_Price_Change",

            "Volatility"
        ]
    ],

    on="Year",

    how="left"

)

In [15]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2083 entries, 0 to 2082
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Country Name       2083 non-null   object 
 1   Year               2083 non-null   int64  
 2   Conflict_Count     2083 non-null   int64  
 3   Average_Intensity  2083 non-null   float64
 4   Country Code_x     603 non-null    object 
 5   Military_Spending  603 non-null    object 
 6   Country Code_y     603 non-null    object 
 7   Inflation_Rate     579 non-null    float64
 8   Average_Oil_Price  1283 non-null   float64
 9   Oil_Price_Change   1249 non-null   float64
 10  Volatility         1249 non-null   float64
dtypes: float64(5), int64(2), object(4)
memory usage: 179.1+ KB


In [16]:
master.isnull().sum()

,0
Country Name,0
Year,0
Conflict_Count,0
Average_Intensity,0
Country Code_x,1480
Military_Spending,1480
Country Code_y,1480
Inflation_Rate,1504
Average_Oil_Price,800
Oil_Price_Change,834


In [17]:
master["Military_Spending"] = master[
    "Military_Spending"
].fillna(0)

master["Inflation_Rate"] = master[
    "Inflation_Rate"
].fillna(0)

In [19]:
master["Military_Spending"] = pd.to_numeric(master["Military_Spending"], errors='coerce').fillna(0)

master["Military_Rank"] = (

    master

    .groupby("Year")["Military_Spending"]

    .rank(

        ascending=False,

        method="dense"

    )

)

In [20]:
master["Inflation_Level"] = pd.cut(

    master["Inflation_Rate"],

    bins=[

        -100,

        2,

        5,

        10,

        1000

    ],

    labels=[

        "Low",

        "Moderate",

        "High",

        "Extreme"

    ]

)

In [21]:
master["Conflict_Severity"] = pd.cut(

    master["Conflict_Count"],

    bins=[

        0,

        5,

        20,

        100,

        1000

    ],

    labels=[

        "Low",

        "Medium",

        "High",

        "Extreme"

    ]

)

In [22]:
master["Oil_Level"] = pd.qcut(

    master["Average_Oil_Price"],

    q=4,

    labels=[

        "Low",

        "Medium",

        "High",

        "Very High"

    ]

)

In [23]:
master.head()

,Country Name,Year,Conflict_Count,Average_Intensity,Country Code_x,Military_Spending,Country Code_y,Inflation_Rate,Average_Oil_Price,Oil_Price_Change,Volatility,Military_Rank,Inflation_Level,Conflict_Severity,Oil_Level
0,Afghanistan,1978,1,2.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,1.0,Low,Low,NaN
1,Afghanistan,1979,1,2.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,1.0,Low,Low,NaN
2,Afghanistan,1980,1,2.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,1.0,Low,Low,NaN
3,Afghanistan,1981,1,2.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,1.0,Low,Low,NaN
4,Afghanistan,1982,1,2.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,1.0,Low,Low,NaN


In [24]:
master.describe(include="all")

,Country Name,Year,Conflict_Count,Average_Intensity,Country Code_x,Military_Spending,Country Code_y,Inflation_Rate,Average_Oil_Price,Oil_Price_Change,Volatility,Military_Rank,Inflation_Level,Conflict_Severity,Oil_Level
count,2083,2083.000000,2083.000000,2083.000000,603,2.083000e+03,603,2083.000000,1283.000000,1249.000000,1249.000000,2083.000000,2083,2083,1283
unique,165,NaN,NaN,NaN,56,NaN,56,NaN,NaN,NaN,NaN,NaN,4,2,4
top,Myanmar (Burma),NaN,NaN,NaN,AFG,NaN,AFG,NaN,NaN,NaN,NaN,NaN,Low,Low,Low
freq,78,NaN,NaN,NaN,25,NaN,25,NaN,NaN,NaN,NaN,NaN,1615,2069,333
mean,NaN,1991.709073,1.351896,1.265731,NaN,6.679950e+09,NaN,3.228667,47.462412,0.843121,6.140898,6.411426,NaN,NaN,NaN
std,NaN,21.374409,0.864238,0.421443,NaN,5.735922e+10,NaN,16.824664,28.182694,14.486361,26.045419,8.802301,NaN,NaN,NaN
min,NaN,1946.000000,1.000000,1.000000,NaN,0.000000e+00,NaN,-12.296984,14.422072,-44.515516,-47.777669,1.000000,NaN,NaN,NaN
25%,NaN,1976.000000,1.000000,1.000000,NaN,0.000000e+00,NaN,0.000000,20.575564,-4.810317,-12.171323,1.000000,NaN,NaN,NaN
50%,NaN,1992.000000,1.000000,1.000000,NaN,0.000000e+00,NaN,0.000000,41.506024,0.201840,0.776812,1.000000,NaN,NaN,NaN
75%,NaN,2010.500000,1.000000,1.500000,NaN,1.245925e+08,NaN,0.811353,68.135100,7.506669,24.909439,11.000000,NaN,NaN,NaN


In [25]:
master.to_csv(

    "global_conflict_master_dataset.csv",

    index=False

)

print("Master Dataset Created Successfully")

Master Dataset Created Successfully
